In [1]:
import ast
import nibabel as nib
import numpy as np
import pandas as pd
import os
from itertools import combinations
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

from Settings import Settings
from Xplainers.evaluations import size_norm_importance

In [2]:
xais = ["BP", "GBP", "IG", "IDGI", "LRP", "OS", "LIME","RISE", "GC++", "SC", "OC", "LC"]

In [ ]:
# load the atlas
atlas_dict1 = {}
atlas_name = "harvard_oxford_cortical"
atlas = nib.load(Settings["atlas_image_path"])
atlas_data = atlas.get_fdata()
with open(f"/opt/home/s4043685/Project1/Resampled_atlases/{atlas_name}_labels.txt", 'r') as file:
    content = file.read()
    region_dict = ast.literal_eval(content)

atlas_dict1["name"] = atlas_name
atlas_dict1["data"] = atlas_data
atlas_dict1["labels"] = region_dict

atlas_dict2 = {}
atlas_name = "harvard_oxford_subcortical"
atlas = nib.load(f"/opt/home/s4043685/Project1/Resampled_atlases/{atlas_name}_atlas.nii.gz")
atlas_data = atlas.get_fdata()
with open(f"/opt/home/s4043685/Project1/Resampled_atlases/{atlas_name}_labels.txt", 'r') as file:
    content = file.read()
    region_dict = ast.literal_eval(content)

atlas_dict2["name"] = atlas_name
atlas_dict2["data"] = atlas_data
atlas_dict2["labels"] = region_dict

# Background image
bg_img = nib.load("Resampled_atlases/bg_img_MNI152lin_1mm.nii.gz")

In [4]:
def idvs_percent(hh_data, atlases):
    idvs = []
    for atlas_dict in atlases:
        atlas_data = atlas_dict["data"]
        for label in np.unique(atlas_data):
            if label == 0:
                continue
            idvs.append(size_norm_importance(atlas_dict, hh_data, label))

    idvs = np.array(idvs, dtype=float)
    total = idvs.sum()
    if total == 0:
        return np.zeros_like(idvs)
    return (idvs / total) * 100

Average Map

In [5]:
for dataset in ["ADNI", "PPMI"]:
    main_folder = f"/opt/home/s4043685/Project1/Explanations/{dataset}_primary"
    for cat in ["TP", "TN"]:
        xai_region_maps = []

        for xai in xais:
            xai_file = f"{main_folder}/Original/{xai}/{cat}.nii.gz"

            hh_img = nib.load(xai_file)
            hh_data = hh_img.get_fdata()

            region_map = np.zeros_like(hh_data, dtype=np.float32)

            for atlas_dict in [atlas_dict1, atlas_dict2]:
                atlas_data = atlas_dict["data"]

                for label in np.unique(atlas_data):
                    if label == 0:
                        continue

                    region_name = atlas_dict["labels"][int(label)]

                    avg_val = size_norm_importance(atlas_dict, hh_data, label)
                    region_map[atlas_data == label] = avg_val

            xai_region_maps.append(region_map)

        final_avg_map = np.mean(xai_region_maps, axis = 0)

        out_file = f"{main_folder}/{dataset}_{cat}_combined_avg_map.nii.gz"
        nib.save(nib.Nifti1Image(final_avg_map, hh_img.affine, hh_img.header), out_file)

Neurobiological Validity

In [ ]:
for dataset in ["ADNI", "PPMI"]:
    main_folder = f"/opt/home/s4043685/Project1/Explanations/{dataset}_primary"

    for cat in ["TP", "TN"]:
        results = {}
        for xai in xais:
            xai_file = f"{main_folder}/Original/{xai}/{cat}.nii.gz"

            hh_data = nib.load(xai_file).get_fdata()
            
            xai_avgs = {}
            for atlas_dict in [atlas_dict1, atlas_dict2]:
                for label in np.unique(atlas_dict["data"]):
                    if label == 0:
                        continue

                    region_name = atlas_dict["labels"][int(label)]
                    avg = size_norm_importance(atlas_dict, hh_data, label)

                    if region_name not in results:
                        results[region_name] = {}

                    results[region_name][xai] = avg

        df = pd.DataFrame.from_dict(results, orient = "index")
        df_percentage = df.div(df.sum(axis = 0), axis = 1) * 100

        df_percentage = df_percentage.sort_index().reset_index()
        df_percentage = df_percentage.rename(columns = {"index": "region"})
        df_percentage.to_csv(f"/opt/home/s4043685/Project1/Explanations/{dataset}_{cat}_pct_imp.csv")

        df_percentage_norm = df_percentage.copy()
        xai_cols = [col for col in df_percentage_norm.columns if col != "region"]
        df_percentage_norm[xai_cols] = (
            df_percentage_norm[xai_cols] - df_percentage_norm[xai_cols].min()
        ) / (
            df_percentage_norm[xai_cols].max() - df_percentage_norm[xai_cols].min()
        )

        df_percentage_norm.to_csv(f"/opt/home/s4043685/Project1/Explanations/{dataset}_{cat}_norm_pct_imp.csv")

        # Inter-method agreement
        numeric_df = df_percentage_norm.drop(columns=["region"])

        spearman_all = numeric_df.corr(method="spearman")

        corr_matrix = spearman_all.values
        labels = spearman_all.columns

        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        masked_corr = np.ma.masked_where(mask, corr_matrix)

        # Plot
        plt.figure()
        plt.imshow(masked_corr, cmap = "bwr")
        plt.xticks(range(len(labels)), labels, rotation=90)
        plt.yticks(range(len(labels)), labels)
        plt.title("Spearman Correlation Matrix (Lower Triangle)")
        plt.colorbar(label="Spearman ρ")
        plt.tight_layout()
        plt.show()

Stability of IDV heatmaps

In [ ]:
for dataset in ["ADNI", "PPMI"]:
    main_folder = f"/opt/home/s4043685/Project1/Explanations/{dataset}_primary"

    rows = []
    for xai in xais:
        xai_folder = f"{main_folder}/Original/{xai}"

        xai_row = {"xai": xai}

        for cat in ["TP", "TN"]:
            cat_folder = f"{xai_folder}/{cat}"
            all_files = os.listdir(cat_folder)

            cat_idvs = []
            for file in all_files:
                file_path = os.path.join(cat_folder, file)
                hh_data = nib.load(file_path).get_fdata()

                idv = idvs_percent(hh_data, [atlas_dict1, atlas_dict2])
                cat_idvs.append(idv)

            corrs = []
            for a, b in combinations(cat_idvs, 2):
                rho, pval = spearmanr(a, b)
                if np.isfinite(rho):
                    corrs.append(rho)

            xai_row[cat] = corrs

        rows.append(xai_row)

    df_corrs = pd.DataFrame(rows)
    df_corrs.to_csv(f"/opt/home/s4043685/Project1/Explanations/{dataset}_idv_corrs.csv")

Class Discrimination

In [16]:
pair_specs = {
    "TP_vs_TN": ("TP", "TN"),
    "TP_vs_FP": ("TP", "FP"),
    "TN_vs_FN": ("TN", "FN"),
    "TP_vs_FN": ("TP", "FN"),
    "TN_vs_FP": ("TN", "FP"),
    "FN_vs_FP": ("FN", "FP"),
}

In [ ]:
for dataset in ["ADNI", "PPMI"]:
    main_path = f"/opt/home/s4043685/Project1/Explanations/{dataset}_primary/Original"
    rows = []
    for xai in xais:
        base = os.path.join(main_path, xai)

        maps = {}
        for key in ["TP", "TN", "FP", "FN"]:
            path = os.path.join(base, f"{key}.nii.gz")
            maps[key] = nib.load(path).get_fdata()

        vecs = {}
        for key, vol in maps.items():
            vecs[key] = idvs_percent(vol, [atlas_dict1, atlas_dict2])

        row = {"xai" : xai}
        for colname, (a, b) in pair_specs.items():
            rho, pval = spearmanr(vecs[a], vecs[b])
            row[colname] = rho

        rows.append(row)

    df_corr = pd.DataFrame(rows)
    df_corr.to_csv(f"/opt/home/s4043685/Project1/Explanations/{dataset}_class_corrs.csv")

Stability with noise

In [ ]:
noise_levels = [
    "gaussian-noise-0.02", "gaussian-noise-0.04", "gaussian-noise-0.06",
    "poisson-noise-40", "poisson-noise-50", "poisson-noise-60",
    "rician-noise-0.02", "rician-noise-0.04", "rician-noise-0.06",
    "gibbs-ringing-0.3", "gibbs-ringing-0.45","gibbs-ringing-0.6",
    "randomization-similar", "randomization-complete"
]

In [ ]:
for dataset in ["ADNI","PPMI"]:
    rows = []
    main_folder = f"/opt/home/s4043685/Project1/Explanations/{dataset}_primary"

    for xai in xais:
        xai_folder = f"{main_folder}/Original/{xai}"
        row = {"xai" : xai}

        for noise in noise_levels:
            noise_folder = f"{main_folder}/{noise}/{xai}"

            for cat in ["TP","TN"]:
                orig_cat = f"{xai_folder}/{cat}"
                noise_cat = f"{noise_folder}/{cat}"

                if not (os.path.isdir(orig_cat) and os.path.isdir(noise_cat)):
                    continue

                orig_files = os.listdir(orig_cat)
                noise_files = os.listdir(noise_cat)

                common_files = sorted(set(orig_files) & set(noise_files))

                corrs = []
                for file in common_files:
                    print(dataset, xai, noise, cat, file)
                    orig_path = os.path.join(orig_cat, file)
                    noise_path = os.path.join(noise_cat, file)

                    orig_data = nib.load(orig_path).get_fdata()
                    noise_data = nib.load(noise_path).get_fdata()

                    orig_idvs = idvs_percent(orig_data, [atlas_dict1, atlas_dict2])
                    noise_idvs = idvs_percent(noise_data, [atlas_dict1, atlas_dict2])

                    rho, pval = spearmanr(orig_idvs, noise_idvs)

                    if np.isfinite(rho):
                        corrs.append(rho)

                row[f"{cat}_{noise}"] = corrs

        rows.append(row)  

    df_corrs = pd.DataFrame(rows)  
    df_corrs.to_csv(f"/opt/home/s4043685/Project1/Explanations/{dataset}_noise_corrs.csv")  

Image Stability

In [ ]:
def psnr(img, noisy_img, mask, max_val=1.0):
    # Apply mask
    img_m = img[mask > 0]
    noisy_img_m = noisy_img[mask > 0]

    mse = np.mean((img_m - noisy_img_m) ** 2)
    if mse == 0:
        return np.inf
    return 10 * np.log10((max_val ** 2) / mse)

In [ ]:
for dataset in ["ADNI", "PPMI"]:
    folder = f"/opt/home/s4043685/Project1/{dataset}"
    files = os.listdir(f"{folder}/test")

    for noise in noise_levels:
        psnrs = []
        
        for sub in files:
            orig_file = os.path.join(f"{folder}/test", sub)
            noise_file = os.path.join(f"{folder}/test_{noise}", sub)

            orig_data = nib.load(orig_file).get_fdata()
            noise_data = nib.load(noise_file).get_fdata()

            orig_data -= orig_data.min()
            orig_data /= orig_data.max()

            noise_data -= noise_data.min()
            noise_data /= noise_data.max()

            psnr_value = psnr(orig_data, noise_data, bg_img.get_fdata(), max_val = 1.0)
            psnrs.append(psnr_value)

            print(dataset, noise, np.mean(psnrs), np.std(psnrs))